# SobolevCellular - offline MoA inference
Internet OFF. Reads bundled weights + calibration and the mounted competition data, writes submission.csv. Build the bundle first: python upload_kaggle_dataset.py.

In [ ]:
# Offline inference kernel (internet OFF). Auto-discovers the bundle + competition data.
import os, sys, glob, json

bundle_root = os.path.dirname(glob.glob("/kaggle/input/*/src")[0])   # dir containing src/
comp_dir    = os.path.dirname(glob.glob("/kaggle/input/*/test.csv")[0])
sys.path.insert(0, bundle_root)
os.environ["CPM_DATA_DIR"]    = comp_dir
os.environ["CPM_WEIGHTS_DIR"] = os.path.join(bundle_root, "weights")

BACKBONE = "imagenet_convnext_tiny"
calib = json.load(open(os.path.join(bundle_root, f"calib_{BACKBONE}.json")))

from src import cv, config
from src.infer import predict, write_submission
from src.train import get_device

rows = cv.load_rows(os.path.join(comp_dir, "test.csv"))
ids, probs = predict(BACKBONE, rows, calib, get_device("auto"))
n = write_submission(ids, probs, "/kaggle/working/submission.csv",
                     os.path.join(comp_dir, "sample_submission.csv"))
print(f"wrote submission.csv: {n} rows, thin-mass mean={probs[:, config.THIN_CLASSES].sum(1).mean():.3f}")
